In [1]:
import torch
from torch.utils.data import DataLoader, Subset
from transformers import T5TokenizerFast, CLIPProcessor, CLIPTokenizerFast, CLIPImageProcessorFast, T5ForConditionalGeneration


from peft import LoraConfig, get_peft_model, TaskType
from torch.amp import autocast, GradScaler
from tqdm import tqdm
import os 


In [2]:
from Modules.config import (TRAIN_IMAGE_DIR,
                            TEST_IMAGE_DIR,
                            FAISS_IMAGE_PATH,
                            TRAIN_METADATA_PATH,
                            TEST_METADATA_PATH,
                            CLIP_MODEL_NAME,
                            T5_MODEL_NAME,
                            VLM_CHECKPOINT_DIR,
                            T5_DECODER_LORA_CONFIG)

from Modules.FusionVLM import FusionVLM, create_default_FusionVLM, load_default_FusionVLM, save_FusionVLM, apply_lora_config
from Modules.retrieval_module import Retriever
from Modules.datasets import VLMDataset, VLMDataCollator
from Modules.utils import print_model_param_stats, add_dict
from Modules.metrics import evaluate_captioning, setup_nltk
from Modules.train_VLM import train_and_evaluate_model

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [4]:
CLIP_processor = CLIPImageProcessorFast.from_pretrained(CLIP_MODEL_NAME, local_files_only=True)
CLIP_tokenizer = CLIPTokenizerFast.from_pretrained(CLIP_MODEL_NAME, local_files_only=True)
T5_tokenizer = T5TokenizerFast.from_pretrained(T5_MODEL_NAME, local_files_only=True)

In [5]:
collator_T5 = VLMDataCollator(CLIP_processor, T5_tokenizer, device=DEVICE)
collator_CLIP = VLMDataCollator(CLIP_processor, CLIP_tokenizer, label_tokenizer=T5_tokenizer, max_seq_len=77, device=DEVICE)

# collator = collator_T5
collator = collator_CLIP

In [6]:
retriever = Retriever(metadata_path=TRAIN_METADATA_PATH, faiss_path=FAISS_IMAGE_PATH)

train_dataset = VLMDataset(image_dir=TRAIN_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TRAIN_METADATA_PATH,
                           retriever=retriever)

test_dataset = VLMDataset(image_dir=TEST_IMAGE_DIR, 
                           ref_image_dir=TRAIN_IMAGE_DIR,
                           metadata_path=TEST_METADATA_PATH,
                           retriever=retriever)

In [7]:
BATCH_SIZE = 16
NUM_WORKERS = 0

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [ ]:
# import numpy as np
# from torch.utils.data import DataLoader, Subset

# num_train_samples = 1024
# num_test_samples = 128


# indices = np.random.choice(len(train_dataset), num_train_samples, replace=False)
# train_subset = Subset(train_dataset, indices)
# train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

# indices = np.random.choice(len(test_dataset), num_test_samples, replace=False)
# test_subset = Subset(test_dataset, indices)
# test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=collator)

In [ ]:
model = create_default_FusionVLM().to(DEVICE)

num_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {num_params:,}\nText Decoder", end=' ')
# model.text_decoder.print_trainable_parameters()

In [8]:
model = FusionVLM(vision_encoder_name=CLIP_MODEL_NAME,
                    text_encoder_name=CLIP_MODEL_NAME,
                    T5_text_decoder_name=T5_MODEL_NAME,
                    num_fusion_blocks=4,
                    use_local_files=True
                    )


In [9]:
model = apply_lora_config(model).to(DEVICE)

c:\Users\Mahan\Documents\Projects\Retrieval-Augmented-Image-Captioning\.venv\Lib\site-packages\peft\tuners\tuners_utils.py:1225: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


In [10]:
print_model_param_stats(model)

Module                                          Total    Trainable       Frozen
--------------------------------------------------------------------------------
vision_encoder                             87,456,000            0   87,456,000
text_encoder                               63,165,952            0   63,165,952
vision_proj                                   590,592      590,592            0
text_proj                                     393,984      393,984            0
fusion_blocks                              56,724,480   56,724,480            0
post_fusion_ln                                  1,536        1,536            0
fusion_proj                                   590,592      590,592            0
text_decoder                              254,655,744   31,752,192  222,903,552
--------------------------------------------------------------------------------
TOTAL                                     463,578,880   90,053,376  373,525,504


In [ ]:
print_model_param_stats(model)

In [11]:
NUM_EPOCHS = 2
full_history = {}
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)

setup_nltk()
os.makedirs(VLM_CHECKPOINT_DIR, exist_ok=True)

In [12]:
history = train_and_evaluate_model(model, train_loader, optimizer, NUM_EPOCHS, test_loader, T5_tokenizer)
add_dict(full_history, history)

Epoch 1:   7%|▋         | 139/1924 [01:05<13:58,  2.13it/s, loss=2.24]


KeyboardInterrupt: 

In [ ]:
save_FusionVLM(model, f'epoch2', VLM_CHECKPOINT_DIR)

In [ ]:
full_history

In [ ]:
import json
with open('history.json', "w", encoding="utf-8") as f:
    json.dump(full_history, f, indent=2, ensure_ascii=False)

In [15]:
with torch.no_grad():
    for batch in test_loader:
        gt_captions = batch["all_captions"]  # List[List[str]]

        generated_ids = model.generate(
            query_pixel_values=batch["query_pixel_values"],
            # retrieved_pixel_values=batch["retrieved_pixel_values"],
            retrieved_pixel_values=batch.get("retrieved_pixel_valuess"),
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            max_length=64,
            num_beams=1,
            # do_sample=True,
            # top_p=0.9,
            # temperature=0.8,
            # repetition_penalty=1.2,
        )

        decoded = T5_tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        for i in range(len(decoded)):
            print(gt_captions[i])
            print(decoded[i])            
        break

['The golfer in the blue shirt and visor is chipping out of the bunker to try to reach the green from roughly 150 yards .', 'A man trying knock his golf ball that is stuck in a sand trap on a golf course .', 'A man takes a swing of his club  with sand coming up out of the dune .', 'Man playing golf  trying to get a golf ball out of the sand .', 'On a beautiful day a man is playing golf .']
A picture of A man with a hat and hats with a dog is a man with a dog in the background is a man in a sandbox .
['A woman sings on stage as a man plays an instrument in unison with her vocals .', 'A blond woman in a burgundy dress is singing  while a man plays a trumpet .', 'There is a duet being performed by a male on a trumpet and a lady singing .', 'A curly headed man is playing an instrument  and a lady in a mauve dress .', 'A blond woman in a long dress sings while a man plays trumpet .']
A picture of A young man with a dog in a white hat a man with a black belt and a
['A man in a white Bobcat s